# Risk Scoring Notebook

This notebook is dedicated to delay risk scoring using probability from a classification model.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
file_path = "/home/pratyush_device/Documents/college/AIML lab/Project/customer_cleaned.csv"
df = pd.read_csv(file_path)
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (750, 26)


,acquisition_cost_usd,order_value_usd,satisfaction_score,support_tickets,lead_time_days,days_acq_to_order,days_order_to_payment,acquisition_date_year,acquisition_date_month,acquisition_date_dayofweek,...,market_segment_Asia-Pacific,market_segment_Europe,market_segment_Middle East,market_segment_North America,supplier_id_SUP-B,supplier_id_SUP-C,supplier_id_SUP-D,supplier_id_SUP-E,supplier_id_SUP-F,supplier_id_SUP-G
0,850,12500,4,1,14,10,34,2024,1,4,...,False,False,False,True,False,False,False,False,False,False
1,920,8500,5,0,16,12,38,2024,1,5,...,False,True,False,False,True,False,False,False,False,False
2,880,21000,4,2,15,13,41,2024,1,6,...,True,False,False,False,False,True,False,False,False,False
3,790,7500,3,1,13,14,37,2024,1,0,...,False,False,False,True,False,False,False,False,False,False
4,950,15000,5,0,12,16,40,2024,1,1,...,False,True,False,False,False,False,True,False,False,False


In [3]:
target_col = "lead_time_days"
if target_col not in df.columns:
    raise ValueError("lead_time_days column not found in customer_cleaned.csv")

# Build delay target from lead_time_days
delay_threshold = df[target_col].quantile(0.75)
df["delay"] = (df[target_col] > delay_threshold).astype(int)

print("Delay threshold:", delay_threshold)
print(df["delay"].value_counts(normalize=True))

Delay threshold: 16.0
delay
0    0.877333
1    0.122667
Name: proportion, dtype: float64


In [4]:
# Prepare model features
X_raw = df.drop(columns=[target_col, "delay"])
X = pd.get_dummies(X_raw, drop_first=True)
y = df["delay"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Confusion Matrix:
[[132   0]
 [  0  18]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       132
           1       1.00      1.00      1.00        18

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



In [5]:
def get_risk(prob):
    if prob > 0.7:
        return "High"
    elif prob > 0.4:
        return "Medium"
    return "Low"


def recommend_action(risk):
    if risk == "High":
        return "Expedite shipment or change supplier"
    if risk == "Medium":
        return "Monitor order closely"
    return "No action needed"

In [6]:
# Risk scoring for all orders
all_prob = clf.predict_proba(X)[:, 1]
scored = X_raw.copy()
scored["Delay_Probability"] = all_prob
scored["Risk_Level"] = scored["Delay_Probability"].apply(get_risk)
scored["Recommended_Action"] = scored["Risk_Level"].apply(recommend_action)

print(scored["Risk_Level"].value_counts())
scored.head(10)

Risk_Level
Low       658
High       90
Medium      2
Name: count, dtype: int64


,acquisition_cost_usd,order_value_usd,satisfaction_score,support_tickets,days_acq_to_order,days_order_to_payment,acquisition_date_year,acquisition_date_month,acquisition_date_dayofweek,order_date_year,...,market_segment_North America,supplier_id_SUP-B,supplier_id_SUP-C,supplier_id_SUP-D,supplier_id_SUP-E,supplier_id_SUP-F,supplier_id_SUP-G,Delay_Probability,Risk_Level,Recommended_Action
0,850,12500,4,1,10,34,2024,1,4,2024,...,True,False,False,False,False,False,False,0.015,Low,No action needed
1,920,8500,5,0,12,38,2024,1,5,2024,...,False,True,False,False,False,False,False,0.280,Low,No action needed
2,880,21000,4,2,13,41,2024,1,6,2024,...,False,False,True,False,False,False,False,0.005,Low,No action needed
3,790,7500,3,1,14,37,2024,1,0,2024,...,True,False,False,False,False,False,False,0.140,Low,No action needed
4,950,15000,5,0,16,40,2024,1,1,2024,...,False,False,False,True,False,False,False,0.010,Low,No action needed
5,830,9500,4,1,18,42,2024,1,2,2024,...,False,False,False,False,True,False,False,0.055,Low,No action needed
6,910,18000,5,0,19,42,2024,1,3,2024,...,True,True,False,False,False,False,False,0.005,Low,No action needed
7,860,11000,4,0,21,42,2024,1,4,2024,...,False,False,False,False,False,True,False,0.015,Low,No action needed
8,900,13500,4,1,23,42,2024,1,5,2024,...,False,False,True,False,False,False,False,0.005,Low,No action needed
9,810,6500,3,2,25,41,2024,1,6,2024,...,False,False,False,False,False,False,True,0.910,High,Expedite shipment or change supplier


In [7]:
# Single order risk prediction helper
def predict_order_risk(input_row):
    input_encoded = pd.get_dummies(input_row, drop_first=True)
    input_encoded = input_encoded.reindex(columns=X.columns, fill_value=0)
    prob = clf.predict_proba(input_encoded)[0][1]
    risk = get_risk(prob)
    return {
        "Delay Probability": float(prob),
        "Risk Level": risk,
        "Recommended Action": recommend_action(risk)
    }

sample = X_raw.iloc[[0]]
predict_order_risk(sample)

{'Delay Probability': 0.015,
 'Risk Level': 'Low',
 'Recommended Action': 'No action needed'}

In [8]:
output_path = "/home/pratyush_device/Documents/college/AIML lab/Project/risk_scored_orders.csv"
scored.to_csv(output_path, index=False)
print("Saved risk scored file to:", output_path)

Saved risk scored file to: /home/pratyush_device/Documents/college/AIML lab/Project/risk_scored_orders.csv
